# NeuraSight — Chest X-Ray Model Training

Production-quality training notebook for chest X-ray classification.
Trains **EfficientNet-B0, ResNet-50, DenseNet-121** with Optuna
hyperparameter tuning. Dataset: Normal / Pneumonia / Tuberculosis.

Saves weights, probabilities, and metrics to Google Drive for the
stacking ensemble meta-learner.

## 1. Setup

In [ ]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

# Download dataset fresh to Colab local SSD (much faster than Drive)
LOCAL_DATA = '/content/chest_xray'
if not os.path.isdir(os.path.join(LOCAL_DATA, 'train')):
    print('  Downloading dataset from Kaggle to local disk...')
    !kaggle datasets download -d muhammadrehan00/chest-xray-dataset -p /content/ --force
    print('  Extracting...')
    with zipfile.ZipFile('/content/chest-xray-dataset.zip', 'r') as zf:
        zf.extractall(LOCAL_DATA)
    os.remove('/content/chest-xray-dataset.zip')
    print('  ✓ Dataset ready on local disk')
else:
    print('  ✓ Dataset already on local disk')

# Use local disk for fast I/O during training
RAW_DIR = LOCAL_DATA
print(f'  Location: {RAW_DIR}')
print(f'  Splits: {[d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))]}')
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(RAW_DIR, split)
    if os.path.isdir(split_dir):
        classes = sorted([c for c in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, c))])
        print(f'  {split}/: {classes}')

In [ ]:
!pip install timm grad-cam seaborn optuna -q

## 2. Imports

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, accuracy_score
)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Configuration

In [ ]:
MODEL_CONFIG = {
    "efficientnet": {"timm_name": "efficientnet_b0",
                     "save_name": "CHEST_XRAY_EFFICIENTNET"},
    "resnet": {"timm_name": "resnet50",
              "save_name": "CHEST_XRAY_RESNET"},
    "densenet": {"timm_name": "densenet121",
                 "save_name": "CHEST_XRAY_DENSENET"},
}

MODELS_TO_TRAIN = ["efficientnet", "resnet", "densenet"]
NUM_CLASSES = 3
CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis"]

# Data directories (pre-split by 00_Dataset_Setup_Chest_Xray.ipynb)
DATA_DIR = RAW_DIR
TRAIN_DIR = os.path.join(RAW_DIR, 'train')
VAL_DIR = os.path.join(RAW_DIR, 'val')
TEST_DIR = os.path.join(RAW_DIR, 'test')

SAVE_DIR = "/content/drive/MyDrive/NeuraSight/models"
REPORTS_DIR = "/content/drive/MyDrive/NeuraSight/reports/chest_xray"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print('Models to train:', MODELS_TO_TRAIN)
print('Data dirs:')
print(f'  Train: {TRAIN_DIR}')
print(f'  Val:   {VAL_DIR}')
print(f'  Test:  {TEST_DIR}')
print(f'Save dir: {SAVE_DIR}')
print(f'Reports dir: {REPORTS_DIR}')

## 4. Exploratory Data Analysis

Examine class distribution, image dimensions, corrupted files,
sample visualizations, and pixel intensity distributions.

In [ ]:
# Gather all image paths and labels from train split for EDA
# Folder names are lowercase: normal, pneumonia, tuberculosis
all_paths, all_labels = [], []
class_counts = {}

# Auto-detect class names from folder structure
FOLDER_CLASSES = sorted(os.listdir(TRAIN_DIR))
print(f'Folder classes (lowercase): {FOLDER_CLASSES}')
print(f'Display CLASS_NAMES: {CLASS_NAMES}')

for cls_idx, cls in enumerate(FOLDER_CLASSES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue
    files = [f for f in os.listdir(cls_dir)
             if f.lower().endswith(('.png','.jpg','.jpeg'))]
    class_counts[cls] = len(files)
    for f in files:
        all_paths.append(os.path.join(cls_dir, f))
        all_labels.append(cls_idx)

print('\nTotal images per class (train):')
for cls, count in class_counts.items():
    print(f'  {cls}: {count}')
print(f'Total: {len(all_paths)}')

In [ ]:
# Class distribution bar chart
plt.figure(figsize=(8, 5))
bars = plt.bar(class_counts.keys(), class_counts.values(),
               color=['#2ecc71', '#e74c3c', '#3498db'])
plt.title('Class Distribution')
plt.xlabel('Class'); plt.ylabel('Number of Images')
for i, (cls, cnt) in enumerate(class_counts.items()):
    plt.text(i, cnt + 10, str(cnt), ha='center',
             fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'class_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Image dimension statistics + corrupted image detection
widths, heights, corrupted = [], [], []

for path in all_paths:
    try:
        img = Image.open(path)
        img.verify()
        img = Image.open(path)  # reopen after verify
        w, h = img.size
        widths.append(w); heights.append(h)
    except Exception:
        corrupted.append(path)

print(f'Corrupted images: {len(corrupted)}')
if corrupted:
    for p in corrupted[:5]: print(f'  {p}')
print(f'\nDimension Stats:')
print(f'  Width  - min:{min(widths)} max:{max(widths)} '
      f'mean:{np.mean(widths):.0f}')
print(f'  Height - min:{min(heights)} max:{max(heights)} '
      f'mean:{np.mean(heights):.0f}')

In [ ]:
# Sample visualization grid (3x3: 3 images per class)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, cls in enumerate(FOLDER_CLASSES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    samples = os.listdir(cls_dir)[:3]
    for col, fname in enumerate(samples):
        img = Image.open(os.path.join(cls_dir, fname))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(f'{cls.capitalize()}', fontsize=12)
        axes[row, col].axis('off')
plt.suptitle('Sample Images (3 per class)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'sample_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pixel intensity distribution per class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, cls in enumerate(FOLDER_CLASSES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    samples = os.listdir(cls_dir)[:50]
    intensities = []
    for fname in samples:
        img = cv2.imread(os.path.join(cls_dir, fname),
                         cv2.IMREAD_GRAYSCALE)
        if img is not None:
            intensities.extend(img.flatten().tolist())
    axes[idx].hist(intensities, bins=50, color='steelblue',
                   alpha=0.7, density=True)
    axes[idx].set_title(f'{cls.capitalize()} Pixel Intensity')
    axes[idx].set_xlabel('Pixel Value')
    axes[idx].set_ylabel('Density')
plt.suptitle('Pixel Intensity Distribution', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'pixel_intensity.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Preparation

Load pre-split train/val/test from Google Drive using ImageFolder.
No manual splitting needed — the dataset is already organized.

In [ ]:
# Remove corrupted images from analysis (already checked in EDA)
# Use ImageFolder to load pre-split train/val/test directly
from torchvision.datasets import ImageFolder

print(f'Loading datasets from pre-split directories...')
print(f'  Train: {TRAIN_DIR}')
print(f'  Val:   {VAL_DIR}')
print(f'  Test:  {TEST_DIR}')
print()
print('Classes will be auto-detected from folder names (lowercase).')
print('Display names remain capitalized: ', CLASS_NAMES)

In [ ]:
# No custom Dataset class needed — ImageFolder handles it
# ImageFolder maps folder names to class indices alphabetically:
# normal=0, pneumonia=1, tuberculosis=2
print('Using torchvision.datasets.ImageFolder')
print('Class-to-index mapping will be: normal->0, pneumonia->1, tuberculosis->2')

In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Load datasets using ImageFolder (pre-split on Drive)
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = ImageFolder(VAL_DIR, transform=val_transform)
test_dataset = ImageFolder(TEST_DIR, transform=val_transform)

# Verify class mapping
print(f'ImageFolder class_to_idx: {train_dataset.class_to_idx}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True, num_workers=2,
                          pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32,
                        shuffle=False, num_workers=2,
                        pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32,
                         shuffle=False, num_workers=2,
                         pin_memory=True)
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## 6. Training Pipeline

Reusable training and evaluation functions.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    '''Train for one epoch, return avg loss and accuracy.'''
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    '''Evaluate model, return avg loss and accuracy.'''
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100.0 * correct / total

In [ ]:
def train_model(model_key, t_loader, v_loader, epochs, lr,
                weight_decay, optimizer_name, scheduler_name):
    '''Training loop for Optuna trials (no saving).'''
    cfg = MODEL_CONFIG[model_key]
    model = timm.create_model(
        cfg['timm_name'], pretrained=True,
        num_classes=NUM_CLASSES
    ).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'adamw':
        opt = optim.AdamW(model.parameters(), lr=lr,
                          weight_decay=weight_decay)
    elif optimizer_name == 'adam':
        opt = optim.Adam(model.parameters(), lr=lr,
                         weight_decay=weight_decay)
    else:
        opt = optim.SGD(model.parameters(), lr=lr,
                        momentum=0.9, weight_decay=weight_decay)

    if scheduler_name == 'cosine':
        sched = optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=epochs, eta_min=1e-6)
    else:
        sched = optim.lr_scheduler.StepLR(
            opt, step_size=5, gamma=0.1)

    best_acc = 0.0
    for epoch in range(epochs):
        train_one_epoch(model, t_loader, opt, criterion, device)
        _, vl_acc = evaluate(model, v_loader, criterion, device)
        sched.step()
        if vl_acc > best_acc:
            best_acc = vl_acc

    return model, best_acc, None

## 7. Hyperparameter Tuning with Optuna

Quick 10-trial study per model (5 epochs each) to find
optimal learning rate, weight decay, optimizer, and scheduler.

In [ ]:
def optuna_objective(trial, model_key):
    '''Optuna objective for hyperparameter search.'''
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)
    bs = trial.suggest_categorical('batch_size', [16, 32])
    opt_name = trial.suggest_categorical(
        'optimizer', ['adam', 'adamw'])
    sched_name = trial.suggest_categorical(
        'scheduler', ['cosine', 'step'])

    t_loader = DataLoader(train_dataset, batch_size=bs,
                          shuffle=True, num_workers=2,
                          pin_memory=True)
    v_loader = DataLoader(val_dataset, batch_size=bs,
                          shuffle=False, num_workers=2,
                          pin_memory=True)

    _, best_acc, _ = train_model(
        model_key, t_loader, v_loader, epochs=5,
        lr=lr, weight_decay=wd,
        optimizer_name=opt_name, scheduler_name=sched_name)
    return best_acc

In [ ]:
# Run Optuna study for each model
best_params = {}

for model_key in MODELS_TO_TRAIN:
    print(f'\n{"="*50}')
    print(f'Optuna Study: {model_key}')
    print(f'{"="*50}')

    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial, mk=model_key: optuna_objective(trial, mk),
        n_trials=10, show_progress_bar=True)

    best_params[model_key] = study.best_params
    print(f'Best acc: {study.best_value:.2f}%')
    print(f'Params: {study.best_params}')

print('\n' + '='*50)
print('OPTUNA RESULTS SUMMARY')
print('='*50)
for key, params in best_params.items():
    print(f'\n{key}:')
    for k, v in params.items():
        print(f'  {k}: {v}')

## 8. Full Training with Best Hyperparameters

Train each model for 15 epochs using Optuna's best params.
Save best checkpoint to Drive when val accuracy improves.

In [ ]:
def full_train_and_save(model_key, params):
    '''Full training with best params, saves all artifacts.'''
    cfg = MODEL_CONFIG[model_key]
    save_name = cfg['save_name']
    best_path = os.path.join(SAVE_DIR, save_name + '.pth')

    batch_size = params.get('batch_size', 32)
    lr = params['lr']
    weight_decay = params['weight_decay']
    optimizer_name = params['optimizer']
    scheduler_name = params['scheduler']
    epochs = 15

    print(f'\nTraining {model_key} for {epochs} epochs')
    print(f'  LR={lr:.6f}, WD={weight_decay:.5f}, '
          f'Opt={optimizer_name}, Sched={scheduler_name}')

    # DataLoaders with tuned batch size
    t_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=2, pin_memory=True
    )
    v_loader = DataLoader(
        val_dataset, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )

    # Build model
    model = timm.create_model(
        cfg['timm_name'], pretrained=True,
        num_classes=NUM_CLASSES
    ).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'adamw':
        optimizer = optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
    elif optimizer_name == 'adam':
        optimizer = optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
    else:
        optimizer = optim.SGD(
            model.parameters(), lr=lr, momentum=0.9,
            weight_decay=weight_decay
        )

    if scheduler_name == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-6
        )
    else:
        scheduler = optim.lr_scheduler.StepLR(
            optimizer, step_size=5, gamma=0.1
        )

    # Training loop
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [],
               'val_loss': [], 'val_acc': []}

    start_time = time.time()
    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(
            model, t_loader, optimizer, criterion, device
        )
        vl_loss, vl_acc = evaluate(
            model, v_loader, criterion, device
        )
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        print(f'  Epoch {epoch+1}/{epochs} | '
              f'Train {tr_acc:.2f}% | Val {vl_acc:.2f}%')

        # Save best checkpoint immediately
        if vl_acc > best_acc:
            best_acc = vl_acc
            torch.save(model.state_dict(), best_path)
            print(f'    -> Saved best ({best_acc:.2f}%)')

    elapsed = time.time() - start_time
    print(f'  Training complete in {elapsed/60:.1f} min')
    print(f'  Best Val Acc: {best_acc:.2f}%')

    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'], label='Val')
    ax1.set_title(f'{model_key} - Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend()

    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'], label='Val')
    ax2.set_title(f'{model_key} - Accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR,
                f'{save_name}_training_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Load best model and compute probabilities on TEST set
    model.load_state_dict(
        torch.load(best_path, map_location=device)
    )
    model.eval()

    probs_list, y_true, y_pred = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)
            probs_list.append(probs.cpu().numpy())
            y_pred.extend(probs.argmax(1).cpu().numpy())
            y_true.extend(labels.numpy())

    probs_arr = np.concatenate(probs_list, axis=0)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Save probabilities and labels
    np.save(os.path.join(SAVE_DIR,
            save_name + '_test_probs.npy'), probs_arr)
    np.save(os.path.join(SAVE_DIR, 'test_labels.npy'), y_true)

    # Compute and save metrics
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro'
    )
    acc = accuracy_score(y_true, y_pred) * 100
    metrics = {
        'model': model_key,
        'timm_name': cfg['timm_name'],
        'accuracy': round(acc, 2),
        'precision': round(prec * 100, 2),
        'recall': round(rec * 100, 2),
        'f1': round(f1 * 100, 2),
        'best_val_acc': round(best_acc, 2),
        'epochs': epochs,
        'best_params': params
    }
    with open(os.path.join(SAVE_DIR,
              save_name + '_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    return metrics, y_true, y_pred

In [ ]:
# Train all models with best hyperparameters
all_metrics = []
all_y_true = None
all_y_preds = {}

for model_key in MODELS_TO_TRAIN:
    print('=' * 60)
    print(f'TRAINING: {model_key}')
    print('=' * 60)
    params = best_params[model_key]
    metrics, y_true, y_pred = full_train_and_save(
        model_key, params)
    all_metrics.append(metrics)
    all_y_true = y_true
    all_y_preds[model_key] = y_pred
    print()

print('\nAll models trained successfully!')

## 9. Evaluation

Classification reports, confusion matrices, and comparison.

In [ ]:
# Per-model evaluation
for model_key in MODELS_TO_TRAIN:
    y_pred = all_y_preds[model_key]
    print(f'\n{"="*50}')
    print(f'{model_key.upper()} - Classification Report')
    print('='*50)
    print(classification_report(
        all_y_true, y_pred, target_names=CLASS_NAMES))

    cm = confusion_matrix(all_y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES)
    plt.title(f'{model_key} Confusion Matrix')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR,
                f'{model_key}_confusion_matrix.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Model comparison table
df = pd.DataFrame(all_metrics)
df = df[['model', 'timm_name', 'accuracy', 'precision',
         'recall', 'f1', 'best_val_acc']]
df = df.sort_values('accuracy', ascending=False).reset_index(
    drop=True)
df.to_csv(os.path.join(REPORTS_DIR, 'model_comparison.csv'),
          index=False)
print('MODEL COMPARISON')
print(df.to_string(index=False))

In [ ]:
# Grouped bar chart
plt.figure(figsize=(10, 6))
metrics_cols = ['accuracy', 'precision', 'recall', 'f1']
x = np.arange(len(df))
width = 0.2
for i, m in enumerate(metrics_cols):
    plt.bar(x + i*width, df[m], width, label=m.capitalize())
plt.xticks(x + 1.5*width, df['model'])
plt.ylabel('Score (%)'); plt.ylim(60, 100)
plt.title('Chest X-Ray Base Model Comparison')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR,
            'model_comparison_chart.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 10. Grad-CAM Visualization

Manual Grad-CAM implementation to visualize which regions
each model focuses on for classification.

In [ ]:
def generate_gradcam(model, img_tensor, target_layer):
    '''Generate Grad-CAM heatmap using hooks.'''
    gradients, activations = [], []

    def fwd_hook(module, inp, out):
        activations.append(out.detach())

    def bwd_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0].detach())

    fh = target_layer.register_forward_hook(fwd_hook)
    bh = target_layer.register_full_backward_hook(bwd_hook)

    model.eval()
    output = model(img_tensor.unsqueeze(0).to(device))
    pred_class = output.argmax(dim=1).item()

    model.zero_grad()
    output[0, pred_class].backward()

    fh.remove(); bh.remove()

    grads = gradients[0].squeeze()   # (C, H, W)
    acts = activations[0].squeeze()   # (C, H, W)
    weights = grads.mean(dim=(1, 2))  # (C,)

    cam = torch.zeros(acts.shape[1:], device=device)
    for i, w in enumerate(weights):
        cam += w * acts[i]
    cam = torch.relu(cam)
    cam = cam - cam.min()
    if cam.max() > 0:
        cam = cam / cam.max()
    cam = cam.cpu().numpy()
    cam = cv2.resize(cam, (224, 224))
    return cam, pred_class


def get_target_layer(model, model_key):
    '''Get last conv layer for Grad-CAM.'''
    if model_key == 'efficientnet':
        return model.conv_head
    elif model_key == 'resnet':
        return model.layer4[-1].conv3
    elif model_key == 'densenet':
        return model.features.denseblock4.denselayer16.conv2
    return None


def denormalize(tensor):
    '''Reverse ImageNet normalization.'''
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = tensor.cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

In [ ]:
# Generate Grad-CAM for each model
for model_key in MODELS_TO_TRAIN:
    cfg = MODEL_CONFIG[model_key]
    save_name = cfg['save_name']
    best_path = os.path.join(SAVE_DIR, save_name + '.pth')

    model = timm.create_model(
        cfg['timm_name'], pretrained=False,
        num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(
        torch.load(best_path, map_location=device))
    target_layer = get_target_layer(model, model_key)

    fig, axes = plt.subplots(3, 2, figsize=(8, 12))
    fig.suptitle(f'Grad-CAM: {model_key}', fontsize=14)

    for cls_idx, cls_name in enumerate(FOLDER_CLASSES):
        cls_dir = os.path.join(TRAIN_DIR, cls_name)
        sample = os.listdir(cls_dir)[0]
        img = Image.open(
            os.path.join(cls_dir, sample)).convert('RGB')
        img_t = val_transform(img)

        cam, pred = generate_gradcam(model, img_t, target_layer)
        orig = denormalize(img_t)

        axes[cls_idx, 0].imshow(orig)
        axes[cls_idx, 0].set_title(
            f'{cls_name.capitalize()} (pred: {FOLDER_CLASSES[pred].capitalize()})')
        axes[cls_idx, 0].axis('off')

        heatmap = cv2.applyColorMap(
            np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(
            heatmap, cv2.COLOR_BGR2RGB).astype(np.float32)/255
        overlay = 0.5 * orig + 0.5 * heatmap
        axes[cls_idx, 1].imshow(overlay)
        axes[cls_idx, 1].set_title('Grad-CAM')
        axes[cls_idx, 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR,
                f'{save_name}_gradcam.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 11. Save All Artifacts

Summary of saved files and download weights.

In [ ]:
# List all saved artifacts
print('Artifacts in SAVE_DIR:')
for fn in sorted(os.listdir(SAVE_DIR)):
    if 'CHEST_XRAY' in fn or fn == 'test_labels.npy':
        fpath = os.path.join(SAVE_DIR, fn)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'  {fn} ({size_mb:.1f} MB)')

print(f'\nReports in REPORTS_DIR:')
for fn in sorted(os.listdir(REPORTS_DIR)):
    print(f'  {fn}')

In [ ]:
from google.colab import files

for key in MODELS_TO_TRAIN:
    sn = MODEL_CONFIG[key]['save_name']
    p = os.path.join(SAVE_DIR, sn + '.pth')
    if os.path.exists(p):
        print(f'Downloading: {sn}.pth')
        files.download(p)

## 12. Model Comparison Summary

All three base models trained and evaluated. Artifacts on Drive:

| File | Description |
|------|-------------|
| `CHEST_XRAY_*.pth` | Model weights |
| `*_test_probs.npy` | Softmax probs on test set |
| `test_labels.npy` | Ground truth labels (test set) |
| `*_metrics.json` | Per-model metrics |
| `model_comparison.csv` | Comparison table |

**Dataset loaded from:** `/content/drive/MyDrive/NeuraSight/datasets/chest_xray/raw`

**Next:** Run `Chest_Xray_Stacking_Ensemble.ipynb` to train
the meta-learner on saved probabilities.